In [1]:
import pandas as pd

### tabular transformer expects data in a DF format

In [3]:
embeddings = "demo/all_n2vplus.emb"

with open(embeddings, 'r') as f:
    first_line = f.readline()
    num_nodes, emb_dim = map(int, first_line.strip().split())

df_emb = pd.read_csv(embeddings, skiprows=1, sep = ' ', header = None)

df_emb.rename(columns={0:'node_id'}, inplace=True)

embedding_cols = [f'emb{i}' for i in range(emb_dim)]
df_emb.solumns = ['node_id'] + embedding_cols

df_emb

/var/folders/ps/w4c6pfcn4hn1n6vbjpq6794w0000gp/T/ipykernel_84365/4235987244.py:12: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_emb.solumns = ['node_id'] + embedding_cols


,node_id,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,128
0,229239,-0.016200,0.005921,0.073419,0.036151,-0.071059,-0.035445,-0.066091,-0.227935,-0.096804,...,0.092027,-0.058864,-0.100250,0.065719,0.123459,0.034667,-0.013363,-0.020001,0.218927,0.209972
1,349960,-0.027514,-0.080078,-0.036934,0.067515,-0.018240,-0.016869,-0.069665,-0.230616,-0.085825,...,0.116182,0.091114,-0.113546,0.100821,0.117250,0.051703,0.012014,0.071052,0.210882,0.211143
2,451828,-0.071351,-0.108720,0.034810,0.150857,0.054920,0.034807,-0.018754,-0.209027,0.020554,...,0.182601,0.076805,-0.107368,0.046390,0.077883,0.035370,-0.045838,-0.046140,0.195715,0.181764
3,298910,0.015234,-0.006220,-0.078138,-0.055389,0.030448,-0.147924,-0.087999,-0.226905,0.048794,...,0.036949,0.095145,-0.094461,0.114426,-0.065201,0.019495,-0.056148,-0.106633,0.222125,-0.018498
4,269305,-0.186198,-0.057258,0.051238,0.042863,-0.033990,-0.022146,-0.098589,-0.231825,-0.086538,...,0.191546,0.141000,-0.110096,0.156406,0.007403,0.052413,-0.039759,-0.019449,0.215584,0.110241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19380,296847,-0.023395,0.007989,-0.017585,-0.065283,-0.023501,-0.072326,-0.025136,-0.147224,-0.088445,...,0.077080,0.024166,-0.084837,0.148708,0.078794,0.010558,-0.038134,-0.042341,0.149819,0.102454
19381,485051,-0.035958,-0.014783,-0.018900,-0.054081,-0.015117,-0.079302,-0.040744,-0.152468,-0.085778,...,0.072487,0.035039,-0.074505,0.171013,0.065142,0.007406,-0.043035,-0.056032,0.144909,0.112644
19382,303234,-0.037598,-0.007079,-0.016183,-0.072046,-0.038592,-0.076570,-0.037133,-0.135817,-0.071843,...,0.056538,0.022857,-0.071420,0.151335,0.063173,0.000718,-0.035855,-0.059651,0.141106,0.103220
19383,358542,-0.026331,-0.002575,0.002062,-0.025596,-0.022679,-0.063513,-0.006885,-0.163545,-0.081539,...,0.071941,0.015422,-0.081423,0.157130,0.076932,0.010787,-0.058393,-0.050583,0.163789,0.116368


### if need to follow og code approach with sampling target nodes

In [7]:
f = open('/Users/eshav/Downloads/bioDBnet_db2db_250707135915_1812803228.txt', 'r')
lines = f.readlines()

ensembl_items = list()
for line in lines[1:]:
    items = line.split('\t')[1]
    if items == '-': continue
    items = items.split(';')
    for item in items:
        item = item.strip()
        try:
            if item != '': ensembl_items.append(int(item.replace('ENSP','')))
        except:
            pass

len(ensembl_items)

4311

In [8]:
import numpy as np
import random
random.seed(42)

vectors = list()
ids = list()
labels = list()

f = open("demo/all_n2vplus.emb", 'r')
lines = f.readlines()

X = []
y = []


for line in lines[1:]:
    line = list(map(float, line.split()))
    key = int(line[0])
    value = np.array(line[1:])
    
    vectors.append(value)
    ids.append(key)
    labels.append(0)
    
    if key in ensembl_items:
        labels[-1] = 1
        X.append(value)
        y.append(1)
    else:
        if random.random() > 0.9:
            X.append(value)
            y.append(0)
vectors = np.array(vectors)
X = np.array(X)

In [9]:
len(ids)

19385

In [10]:
key2val = {i:v for i, v in zip(ids, vectors)}

In [11]:
print(f'# of targets: {y.count(1)}\n# of non-targets: {y.count(0)}\n# of dataset: {len(y)}')

# of targets: 607
# of non-targets: 1878
# of dataset: 2485


In [12]:
f = open('/Users/eshav/Downloads/bioDBnet_db2db_250707135915_1812803228.txt', 'r')
lines = f.readlines()

n = 0

ensembl_items = list()
for line in lines[1:]:
    items = line.split('\t')[1]
    if items == '-': continue
    items = items.split(';')
    
    for item in items:
        item = item.strip()
        try:
            if item != '':
                if int(item.replace('ENSP','')) in ids:
                       n += 1
                       break
        except:
            pass

print(n)

601


In [13]:
import pandas as pd
df = pd.DataFrame(X)
df['target'] = y
df

,0,1,2,3,4,5,6,7,8,9,...,119,120,121,122,123,124,125,126,127,target
0,-0.071351,-0.108720,0.034810,0.150857,0.054920,0.034807,-0.018754,-0.209027,0.020554,-0.154530,...,0.076805,-0.107368,0.046390,0.077883,0.035370,-0.045838,-0.046140,0.195715,0.181764,1
1,0.038651,-0.017476,0.052701,0.071277,0.014295,-0.158209,-0.049758,-0.114217,-0.226067,-0.111746,...,0.058139,-0.179301,0.077051,0.098540,0.085674,0.062816,0.011521,0.223031,0.226825,1
2,-0.077355,-0.047118,-0.050669,0.057968,-0.087158,0.041584,0.037871,-0.192774,-0.132135,-0.073415,...,0.084698,-0.090835,0.194844,0.120005,0.036501,0.019904,-0.113406,0.325233,0.170783,1
3,-0.007166,-0.154761,0.013473,0.160234,0.054655,-0.238443,-0.055751,-0.151960,-0.110707,-0.179474,...,0.174864,-0.211960,0.172678,0.170753,0.102318,0.021428,0.021918,0.247352,0.175978,1
4,-0.069395,-0.056747,0.016273,0.061387,-0.080987,-0.101352,-0.096733,-0.230205,0.075670,-0.135979,...,-0.000274,-0.107589,0.138324,0.125535,0.030488,0.110164,0.046268,0.306246,0.060327,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,-0.004532,-0.017612,-0.010215,-0.048133,0.027408,-0.093663,-0.023460,-0.179353,-0.130757,-0.064173,...,0.044696,-0.097239,0.175149,0.084364,-0.000418,-0.065761,-0.083872,0.175156,0.135987,0
2481,-0.024833,-0.012721,-0.010489,-0.063792,-0.037726,-0.088465,-0.037095,-0.162964,-0.081691,-0.042740,...,0.014041,-0.077851,0.182687,0.095439,0.006484,-0.032868,-0.042720,0.193313,0.108725,0
2482,-0.032563,0.003700,-0.011998,-0.059191,-0.027288,-0.075701,-0.042868,-0.168035,-0.079817,-0.041785,...,0.009477,-0.075682,0.172720,0.083884,0.003583,-0.032967,-0.062465,0.165859,0.110948,0
2483,-0.053250,-0.021628,0.005761,-0.054385,-0.033163,-0.077569,-0.032175,-0.194938,-0.064691,-0.010334,...,-0.011759,-0.055349,0.154603,0.067859,0.018905,-0.064310,-0.052050,0.177940,0.127326,0


### tab transformer

In [20]:
# want to test on just one embedding for now
import torch

embedding_vector = [
    -0.027513668, -0.080077834, -0.036934257, 0.06751499, -0.018239737, -0.016869193, -0.069665335,
    -0.23061633, -0.08582457, -0.04465857, 0.26991144, 0.12853529, -0.03370237, 0.08839588, 0.031650722,
    0.082820885, 0.02376529, 0.049307942, 0.14376765, 0.02657407, 0.04451987, -0.10731577, 0.051517222,
    0.047221567, 0.18596086, -0.0806292, -0.006927808, 0.1105699, 0.07794871, 0.09449251, -0.08247682,
    0.03621001, -0.111233994, 0.22687455, 0.054903574, 0.001763601, -0.084283166, -0.106812485,
    0.08285017, -0.02768451, 0.23949376, 0.06961758, -0.043381833, -0.09586973, -0.065044634, -0.08373077,
    -0.16840506, -0.016097225, 0.07836695, -0.097287856, -0.034121525, -0.0667902, -0.16081211,
    -0.1118682, 0.063373625, -0.02453496, 0.24002972, -0.028980842, 0.030314645, 0.08967312, 0.048401084,
    0.18867286, 0.11915356, -0.024783053, 0.0099872295, -0.04311837, -0.18035375, -0.048660662,
    -0.24939592, 0.002933954, -0.055405788, -0.024191974, 0.06912919, 0.06653469, 0.04849147, 0.09616973,
    -0.04157431, 0.069670476, -0.14980182, -0.060680937, -0.05016552, -0.10617397, 0.024143182,
    -0.23576763, 0.12144609, 0.12171757, -0.07829768, -0.08532, -0.05209472, -0.041155037, 0.1737875,
    -0.10853065, 0.049161773, 0.042523656, 0.2686651, -0.051824443, 0.045311302, -0.024539914, 0.014296686,
    -0.297166, 0.069251634, 0.020476138, -0.05986543, 0.0587506, 0.050429672, -0.07095182, -0.13849694,
    0.05841327, 0.20771608, -0.15344387, -0.18527569, 0.07882007, -0.17395978, 0.2241605, 0.00433421,
    -0.04163309, -0.30042312, 0.027079811, 0.11618183, 0.09111364, -0.11354593, 0.10082091, 0.11725002,
    0.051703375, 0.012014371, 0.07105183, 0.2108817, 0.21114329
]

In [21]:
x_cont = torch.tensor(embedding_vector).unsqueeze(0)  # shape = (1, 128)
x_categ = torch.empty(1, 0).long()  # no categorical features

In [24]:
import torch
import torch.nn as nn
from tab_transformer_pytorch import TabTransformer

cont_mean_std = None

model = TabTransformer(
    categories = (),      # tuple containing the number of unique values within each category
    num_continuous = 128,                # number of continuous values
    dim = 32,                           # dimension, paper set at 32
    dim_out = 1,                        # binary prediction, but could be anything
    depth = 6,                          # depth, paper recommended 6
    heads = 8,                          # heads, paper recommends 8
    attn_dropout = 0.1,                 # post-attention dropout
    ff_dropout = 0.1,                   # feed forward dropout
    mlp_hidden_mults = (4, 2),          # relative multiples of each hidden dimension of the last mlp to logits
    mlp_act = nn.ReLU(),                # activation for final mlp, defaults to relu, but could be anything else (selu etc)
    continuous_mean_std = cont_mean_std # (optional) - normalize the continuous values before layer norm
)

model.eval()

with torch.no_grad():
    pred = model(x_categ, x_cont)
print(f"Prediction: {pred.item():.4f}")

Prediction: 0.0829
